### PCA + Random Forest

Getting the data in the right formata

In [ ]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import joblib

In [2]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from data_loaders import Training_set, build_cases

In [3]:
# uploading the data
ts = Training_set()
ts.create_x()

# sorting the data cronologically
X_all = ts.x.sort_index()
Y_all = ts.y.sort_index()

print(f"Range disponibile in Y: {Y_all.index.min()} ---> {Y_all.index.max()} ({len(Y_all)} casi)")

# split 80% train / 20% test
split_idx = int(len(Y_all) * 0.8)
Y_train_raw = Y_all.iloc[:split_idx]
# leaving 7 days buffer (to avoid overlap)
Y_test_raw  = Y_all.iloc[split_idx + 7:]

x_train, y_train = build_cases(X_all, Y_train_raw)
x_test,  y_test  = build_cases(X_all, Y_test_raw)

print(f"Train cases: {x_train.shape[0]} | Test cases: {x_test.shape[0]}")
print(f"Feature originali: {x_train.shape[1]}")

Range disponibile in Y: 2015-01-01 00:00:00+00:00 ---> 2022-12-31 00:00:00+00:00 (2922 casi)
Train cases: 2337 | Test cases: 578
Feature originali: 3243


### PCA

In [4]:
# need to scale the data before processing
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train.to_numpy(dtype=np.float32))
x_test_scaled  = scaler.transform(x_test.to_numpy(dtype=np.float32))

# cleannig for constant columns
x_train_scaled = np.nan_to_num(x_train_scaled, nan=0.0)
x_test_scaled  = np.nan_to_num(x_test_scaled, nan=0.0)

In [5]:
# pca
pca = PCA(n_components=0.98, svd_solver='full', random_state=42)
x_train_pca = pca.fit_transform(x_train_scaled)
x_test_pca  = pca.transform(x_test_scaled)

In [ ]:
print(f"Feature after PCA (98% variance): {x_train_pca.shape[1]} components")

Feature dopo PCA (98% varianza): 58 componenti


### Random Forest

In [ ]:
def extract_positive_probs(proba_list, y_train):
    """Estrae la probabilità della classe 1 (ON) per tutti i 2352 target."""
    n_samples = proba_list[0].shape[0]
    n_targets = len(proba_list)
    probs_2d = np.zeros((n_samples, n_targets), dtype=np.float32)

    for i, p in enumerate(proba_list):
        if p.shape[1] == 2:
            probs_2d[:, i] = p[:, 1]
        else:
            only_class = y_train.iloc[:, i].iloc[0]
            probs_2d[:, i] = float(only_class)
            
    return probs_2d

In [ ]:
rf_pca = RandomForestClassifier(
    n_estimators=150,
    max_depth=14,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features="sqrt",
    max_samples=0.8,
    n_jobs=-1,
    random_state=42,
    verbose=1
)

print("\nAddestramento Random Forest sulle componenti PCA...")
with joblib.parallel_backend('threading', n_jobs=-1):
    rf_pca.fit(x_train_pca, y_train.to_numpy())

# 5. Predizione delle probabilità e valutazione Micro ROC-AUC
raw_preds = rf_pca.predict_proba(x_test_pca)
y_pred_proba = extract_positive_probs(raw_preds, y_train)

micro_auc = roc_auc_score(y_test.to_numpy(), y_pred_proba, average="micro")
print(f"\n==========================================")
print(f"PCA + Random Forest Test ROC-AUC (micro): {micro_auc:.4f}")
print(f"==========================================")